In [ ]:
with push_base as (

    select
        contact_id,
        to_date(substr(campaign_name, 1, 8), 'YYYYMMDD') as push_dt,
        is_sent,
        is_delivered,
        is_opened

    from push_act

),

first_push_dt as (

    select
        contact_id,
        min(push_dt) as first_push_dt

    from push_base

    group by contact_id

),

first_push as (

    select
        pb.contact_id,
        fp.first_push_dt,

        max(case when pb.is_sent then 1 else 0 end) as is_sent_flg,
        max(case when pb.is_delivered then 1 else 0 end) as is_delivered_flg,
        max(case when pb.is_opened then 1 else 0 end) as is_opened_flg

    from push_base pb

    join first_push_dt fp
        on pb.contact_id = fp.contact_id
        and pb.push_dt = fp.first_push_dt

    group by
        pb.contact_id,
        fp.first_push_dt

),

subscr_min as (

    select
        contact_id,
        min(subscr_date_act::date) as first_subscr_date

    from subscr_status

    group by contact_id

),

base as (

    select
        cc.client_id,
        cc.campaigns_cnt,

        coalesce(fp.is_sent_flg, 0) as is_sent_flg,
        coalesce(fp.is_delivered_flg, 0) as is_delivered_flg,
        coalesce(fp.is_opened_flg, 0) as is_opened_flg,

        case
            when coalesce(fp.is_opened_flg, 0) = 1
             and (
                    sm.first_subscr_date is null
                    or sm.first_subscr_date < fp.first_push_dt
                 )
            then 1
            else 0
        end as opened_not_subscr_flg

    from client_cohorts cc

    left join first_push fp
        on cc.client_id = fp.contact_id

    left join subscr_min sm
        on cc.client_id = sm.contact_id

)

select
    campaigns_cnt,

    count(distinct client_id) as total_clients,

    count(distinct case when is_sent_flg = 1 then client_id end) as sent_clients,
    count(distinct case when is_delivered_flg = 1 then client_id end) as delivered_clients,
    count(distinct case when is_opened_flg = 1 then client_id end) as opened_clients,
    count(distinct case when opened_not_subscr_flg = 1 then client_id end) as opened_not_subscr_clients,

    round(
        count(distinct case when is_sent_flg = 1 then client_id end) * 100.0
        / nullif(count(distinct client_id), 0),
        1
    ) as sent_pct,

    round(
        count(distinct case when is_delivered_flg = 1 then client_id end) * 100.0
        / nullif(count(distinct client_id), 0),
        1
    ) as delivered_pct,

    round(
        count(distinct case when is_opened_flg = 1 then client_id end) * 100.0
        / nullif(count(distinct client_id), 0),
        1
    ) as opened_pct,

    round(
        count(distinct case when opened_not_subscr_flg = 1 then client_id end) * 100.0
        / nullif(count(distinct case when is_opened_flg = 1 then client_id end), 0),
        1
    ) as opened_not_subscr_pct_from_opened,

    round(
        count(distinct case when opened_not_subscr_flg = 1 then client_id end) * 100.0
        / nullif(count(distinct client_id), 0),
        1
    ) as opened_not_subscr_pct_total

from base

group by campaigns_cnt

order by campaigns_cnt;

In [ ]:
display(
    push_result
    .rename(columns={
        'campaigns_cnt': 'Количество<br>кампаний',
        'total_clients': 'Всего<br>клиентов',
        'sent_clients': 'Отправлены<br>пуши',
        'delivered_clients': 'Доставлены<br>пуши',
        'opened_clients': 'Открыли<br>пуши',
        'opened_not_subscr_clients': 'Открыли пуш,<br>но не подписались',
        'sent_pct': 'Доля<br>отправленных, %',
        'delivered_pct': 'Доля<br>доставленных, %',
        'opened_pct': 'Доля<br>открывших, %',
        'opened_not_subscr_pct_from_opened': 'Доля без подписки<br>среди открывших, %',
        'opened_not_subscr_pct_total': 'Доля без подписки<br>от всех клиентов, %'
    })
    .style
    .format({
        'Всего<br>клиентов': '{:,.0f}',
        'Отправлены<br>пуши': '{:,.0f}',
        'Доставлены<br>пуши': '{:,.0f}',
        'Открыли<br>пуши': '{:,.0f}',
        'Открыли пуш,<br>но не подписались': '{:,.0f}',
        'Доля<br>отправленных, %': '{:.1f}%',
        'Доля<br>доставленных, %': '{:.1f}%',
        'Доля<br>открывших, %': '{:.1f}%',
        'Доля без подписки<br>среди открывших, %': '{:.1f}%',
        'Доля без подписки<br>от всех клиентов, %': '{:.1f}%'
    })
)